# Задачник PySpark: От Стажера до Мидла

Вас приветствуют ваши менторы (три независимых ИИ-агента, запущенных специально для вашего обучения):
* **Профессор Ричард Фейнман (Теория):** Раскрывает суть операций через математику, топологию и логику распределенных систем.
* **Senior Big Data Analyst (Инженерия):** Учит писать оптимизированный production-код, понимать планы выполнения и избегать падения кластера.
* **ML Researcher (Применение):** Показывает, как эти трансформации превращаются в качественные темпоральные и статические признаки (фичи) для моделей машинного обучения.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("InternWorkbook") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

# Генерация данных для демонстраций и задач
data = [
    ("u1", "2023-10-01 10:00:00", "electronics", 1500.0, "web"),
    ("u1", "2023-10-02 11:30:00", "groceries", 50.0, "app"),
    ("u1", "2023-10-02 11:35:00", "groceries", 20.0, "app"),
    ("u1", "2023-10-04 09:00:00", "clothing", 300.0, "web"),
    ("u2", "2023-10-01 09:00:00", "clothing", 200.0, "app"),
    ("u2", "2023-10-05 14:00:00", "electronics", 800.0, "web"),
    ("u3", "2023-10-03 16:00:00", "clothing", 120.0, "app"),
    ("u3", "2023-10-03 16:05:00", "clothing", 30.0, "app"),
    ("u3", "2023-10-10 12:00:00", "electronics", 1200.0, "web"),
]
schema = ["user_id", "timestamp", "category", "amount", "platform"]
df = spark.createDataFrame(data, schema)
df = df.withColumn("timestamp", F.to_timestamp("timestamp"))
df.show()


## Раздел 1. Select и Filter
**Профессор Фейнман:** Мы находимся в пространстве узких (narrow) трансформаций. Каждая запись независима. Это вычислительно дешевые $O(N)$ локальные операции без сетевого обмена.
**Senior Big Data Analyst:** Напоминаю про механизм **Predicate Pushdown**. Чтение из колоночных форматов (Parquet/ORC) отбросит целые физические блоки данных, если фильтр строгий. Применяйте `filter` до любых `groupBy` или `join`.

### Демонстрация базовых команд

In [ ]:
# ДЕМОНСТРАЦИОННЫЙ БЛОК
# .filter() - фильтрует строки по условию (аналог WHERE в SQL)
# .select() - оставляет только указанные колонки
demo_filter_df = df.filter(F.col("amount") > 500).select("user_id", "amount")
demo_filter_df.show()

**Задача 1.1:** Отфильтруйте транзакции платформы `app` с суммой `amount < 100`. Оставьте только колонки `user_id`, `category` и `platform`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
task1_df = df # ...


## Раздел 2. Агрегации и GroupBy
**Профессор Фейнман:** Агрегация — это проекция множества элементов партиции на один скаляр или вектор. Для этого система обязана выполнить **Shuffle**: топологическую пересылку данных (All-to-All) по кластеру, чтобы все строки с одним ключом группировки (например, одним пользователем) попали на один физический узел.

**Senior Big Data Analyst:** Вызов метода `groupBy("col_name")` подготавливает данные к шаффлу, группируя их по ключу. За ним должен следовать метод `.agg(...)`, внутри которого перечисляются функции агрегации из пакета `pyspark.sql.functions`.

**Объяснение команд агрегации:**
* `F.sum("col")` — Считает математическую сумму значений. Поддерживает Map-Side Combine (частичная сумма на экзекьюторе до сети), очень быстрая операция.
* `F.avg("col")` — Считает среднее арифметическое. Аналогично, имеет высокую производительность.
* `F.count("col")` — Считает количество строк (не-null значений).
* `F.approx_count_distinct("col")` — **Критически важно!** Приблизительно считает количество уникальных значений алгоритмом HyperLogLog. Используйте всегда, когда допустима погрешность 2-5%, так как `F.countDistinct` требует гигантского расхода RAM и жесткого шаффла всех ключей.
* `F.collect_list("col")` — Собирает значения из группы в обычный массив (array). Потенциально опасно: если одному ключу соответствует 10 млн записей (Data Skew), экзекьютор упадет с OOM (Out Of Memory).

**ML Researcher:** С помощью `avg` и `sum` мы строим статические профили покупателей (LTV, средний чек). А через `collect_list` генерируем историю (sequence) для рекуррентных нейросетей.

In [ ]:
# ДЕМОНСТРАЦИОННЫЙ БЛОК: GroupBy и Агрегации
demo_agg_df = df.groupBy("platform").agg(
    F.sum("amount").alias("total_vol"),                  # Суммарный объем
    F.approx_count_distinct("user_id").alias("uniq_usr") # Уникальные пользователи (приблизительно)
)
demo_agg_df.show()

**Задача 2.1:** Сгруппируйте данные по `category`. Найдите общую выручку (sum) и средний чек (avg). Переименуйте результирующие колонки в `total_revenue` и `avg_ticket`.

In [ ]:
# ВАШ КОД ЗДЕСЬ


**Задача 2.2:** Сгруппируйте данные по `user_id`. Посчитайте количество совершенных транзакций (`count`) и соберите список категорий, в которых он покупал (`collect_list`).

In [ ]:
# ВАШ КОД ЗДЕСЬ


## Раздел 3. Оконные функции (Window Functions)
**Профессор Фейнман:** Если агрегация схлопывает пространство $N$ строк в размерность $K$ уникальных ключей, то оконная функция сохраняет изначальную размерность $N$. Каждая строка остается на месте, но через "окно" она получает доступ к информации о своей локальной окрестности (партиции).

**Senior Big Data Analyst:** Работа с окнами состоит из двух шагов: определения спецификации окна (`WindowSpec`) и применения её через метод `.over()`.

**Объяснение параметров окна (WindowSpec):**
* `Window.partitionBy("col")` — Делит данные на изолированные блоки (партиции). Строка видит только строки с таким же значением `col`.
* `Window.orderBy("col")` — Задает порядок сортировки внутри партиции. Без этого не работают сдвиги (lag/lead) и ранжирование.
* `rowsBetween(start, end)` — Задает физические рамки (размер) окна относительно текущей строки. Например, `rowsBetween(-1, 0)` означает: "сосед сверху и текущая строка". 
  * Вспомогательные переменные: `Window.unboundedPreceding` (самое начало партиции), `Window.currentRow` (текущая).

**ВНИМАНИЕ:** Если вы используете `orderBy`, но не пишете `rowsBetween`, Spark автоматически задаст фрейм: `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. Это заставит любые агрегации (сумма, среднее) работать как нарастающий итог (cumulative)!

**Объяснение метода .over() и оконных функций:**
* `F.sum("col").over(window_spec)` — метод `.over()` работает как триггер. Он говорит Spark: "Примени функцию `sum`, но не глобально, а используя рамки окна `window_spec`". 
* `F.row_number()` — Просто нумерует строки по порядку (1, 2, 3...) внутри партишена.
* `F.dense_rank()` — Ранжирует значения, присваивая одинаковый ранг одинаковым значениям, без пропусков (1, 2, 2, 3).
* `F.lag("col", offset)` — Берет значение из предыдущей строки на расстоянии `offset`. Незаменимо для вычисления разниц во времени.

**ML Researcher:** С помощью окон мы генерируем динамические признаки: "сумма трат за последние N операций", "время после предыдущего захода". Остерегайтесь функции `lead()` (следующая строка), она порождает Data Leakage, заглядывая в будущее.

In [ ]:
# ДЕМОНСТРАЦИОННЫЙ БЛОК: Оконные функции
# Спецификация: партиция по пользователю, сортировка по времени
demo_window = Window.partitionBy("user_id").orderBy("timestamp")

demo_win_df = df.withColumn(
    "cumulative_sum", F.sum("amount").over(demo_window)  # Нарастающий итог из-за дефолтного фрейма!
).withColumn(
    "txn_number", F.row_number().over(demo_window)       # Порядковый номер транзакции пользователя
)
demo_win_df.select("user_id", "timestamp", "amount", "cumulative_sum", "txn_number").show()

**Задача 3.1 (Ранжирование):** Для каждого пользователя (`user_id`) отранжируйте его транзакции по `amount` **по убыванию** (используйте `F.col("amount").desc()`). Примените функцию `F.dense_rank()`. Результат запишите в колонку `amount_rank`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
window_rank = Window # ...


**Задача 3.2 (Сдвиги во времени):** Посчитайте разницу во времени (в секундах) между текущей и предыдущей транзакцией для каждого пользователя. 
*Подсказка: используйте `F.lag("timestamp", 1)` с окном, отсортированным по `timestamp`. Разницу в секундах можно получить, приведя колонки к типу long: `F.col("timestamp").cast("long")`.*

In [ ]:
# ВАШ КОД ЗДЕСЬ
window_lag = Window # ...


**Задача 3.3 (Скользящее окно - Хард):** Для каждого пользователя посчитайте скользящее среднее значение `amount` **строго по текущей и одной предыдущей транзакции** (rolling 2). 
*Подсказка: вам нужно явно ограничить фрейм окна через `.rowsBetween(-1, Window.currentRow)`.*

In [ ]:
# ВАШ КОД ЗДЕСЬ
window_rolling = Window # ...


## Решения (Скрыто)
Чтобы проверить себя, раскомментируйте код.

In [ ]:
## РЕШЕНИЯ ##

# Задача 1.1
# df_task_1 = df.select("user_id", "category", "platform").filter((F.col("platform") == "app") & (F.col("amount") < 100))

# Задача 2.1
# df_task_2_1 = df.groupBy("category").agg(F.sum("amount").alias("total_revenue"), F.avg("amount").alias("avg_ticket"))

# Задача 2.2
# df_task_2_2 = df.groupBy("user_id").agg(F.count("category").alias("txn_count"), F.collect_list("category").alias("categories"))

# Задача 3.1
# window_rank = Window.partitionBy("user_id").orderBy(F.col("amount").desc())
# df_task_3_1 = df.withColumn("amount_rank", F.dense_rank().over(window_rank))

# Задача 3.2
# window_lag = Window.partitionBy("user_id").orderBy("timestamp")
# df_task_3_2 = df.withColumn("prev_time", F.lag("timestamp", 1).over(window_lag)) \
#                 .withColumn("diff_sec", F.col("timestamp").cast("long") - F.col("prev_time").cast("long"))

# Задача 3.3
# window_rolling = Window.partitionBy("user_id").orderBy("timestamp").rowsBetween(-1, Window.currentRow)
# df_task_3_3 = df.withColumn("rolling_2_avg", F.avg("amount").over(window_rolling))
